# Week 08 — Advanced Python | Weekly Review

Review every Day 1-6 topic below, run the self-check cells, then take the
interactive quiz in your terminal:

```bash
python weekly_quiz.py     # 10 MCQ + 5 code challenges, scored /15
```

---

## Day 1: Decorators

```python
import functools

def timed(func):
    @functools.wraps(func)          # preserves __name__ / __doc__
    def wrapper(*args, **kwargs):
        ...                          # before
        result = func(*args, **kwargs)
        ...                          # after
        return result
    return wrapper

@timed                               # sugar for: slow = timed(slow)
def slow(): ...
```

| Concept | Meaning |
|---------|---------|
| `@deco` | `func = deco(func)` |
| `functools.wraps` | keeps the original function's metadata on the wrapper |
| stacked decorators | applied bottom-up: `@a` over `@b` → `a(b(func))` |
| decorator factory | `@repeat(3)` — a function that RETURNS a decorator |

In [ ]:
# Self-check: a decorator that counts calls
import functools

def counted(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        wrapper.calls += 1
        return func(*args, **kwargs)
    wrapper.calls = 0
    return wrapper

@counted
def greet(name: str) -> str:
    """Say hello."""
    return f"Hello, {name}!"

print(greet("Ada"), greet("Bob"))
print("calls:", greet.calls)        # 2
print("metadata kept:", greet.__name__, "-", greet.__doc__)

## Day 2: Iterators &nbsp;·&nbsp; Day 3: Generators

```python
class Countdown:                      # ITERATOR: __iter__ + __next__
    def __init__(self, n): self.n = n
    def __iter__(self): return self
    def __next__(self):
        if self.n <= 0:
            raise StopIteration       # how for-loops know to stop
        self.n -= 1
        return self.n + 1

def countdown(n):                     # GENERATOR: same thing, way less code
    while n > 0:
        yield n
        n -= 1
```

| Concept | Meaning |
|---------|---------|
| iterable | has `__iter__` — can be looped (list, str, dict...) |
| iterator | also has `__next__` — produces values one at a time |
| `StopIteration` | raised when exhausted; `for` catches it silently |
| generator function | any `def` containing `yield`; calling it runs NO code yet |
| generator expression | `(x*x for x in data)` — lazy, no list in memory |
| `yield from` | delegate to another iterable/generator |

In [ ]:
# Self-check: lazy pipeline — nothing runs until consumed
def numbers():
    for n in range(1, 11):
        print(f"  producing {n}")
        yield n

evens = (n for n in numbers() if n % 2 == 0)   # generator expr over a generator
squares = (n * n for n in evens)

print("pipeline built — nothing produced yet!")
print("first 2 squares of evens:")
print(next(squares), next(squares))            # only produces 1,2,3,4 — lazy!

## Day 4: Context Managers

```python
class Timer:
    def __enter__(self): ...; return self      # setup
    def __exit__(self, exc_type, exc, tb): ... # cleanup — ALWAYS runs

from contextlib import contextmanager

@contextmanager
def timer():
    start = time.perf_counter()   # __enter__ part
    yield                          # with-block runs here
    print(time.perf_counter() - start)   # __exit__ part
```

| Concept | Meaning |
|---------|---------|
| `with obj as x:` | `x = obj.__enter__()`; `__exit__` guaranteed on exit |
| `__exit__` returning True | suppresses the exception |
| `@contextmanager` | code before `yield` = setup, after = cleanup |
| `contextlib.suppress(E)` | ignore exception type E inside the block |

In [ ]:
# Self-check: cleanup runs even when the block raises
from contextlib import contextmanager

@contextmanager
def announced(name: str):
    print(f"enter {name}")
    try:
        yield name
    finally:
        print(f"exit {name}")      # runs no matter what

with announced("safe block") as label:
    print(f"inside {label}")

try:
    with announced("crashing block"):
        raise ValueError("boom")
except ValueError as e:
    print(f"caught: {e} — but exit still printed above!")

## Day 5: Type Hints &nbsp;·&nbsp; Day 6: Testing with pytest

```python
def find_user(user_id: int) -> str | None: ...          # Optional return
def apply(x: int, op: Callable[[int], int]) -> int: ...  # function argument

# pytest — plain asserts, discovered from test_* names:
def test_add():
    assert add(2, 3) == 5

def test_rejects_zero():
    with pytest.raises(ValueError):
        add_item({}, "apple", 0)

@pytest.mark.parametrize("n, expected", [(2, 4), (3, 9)])
def test_square(n, expected):
    assert square(n) == expected
```

| Concept | Meaning |
|---------|---------|
| `Optional[X]` / `X \| None` | X or None — check with mypy, not at runtime |
| `list[int]`, `dict[str, int]` | built-in generics (3.9+) |
| `TypeVar` + `Generic` | type-safe code that works with any type |
| pytest discovery | `test_*.py` files, `test_*` functions |
| fixture | setup injected into tests by parameter name |
| parametrize | one test body, many reported cases |

In [ ]:
# Self-check: typed function + pytest-style tests run manually
def average(numbers: list[float]) -> float | None:
    if not numbers:
        return None                # Optional return — hinted honestly
    return sum(numbers) / len(numbers)

def test_average():
    assert average([2.0, 4.0]) == 3.0

def test_average_empty():
    assert average([]) is None

test_average()
test_average_empty()
print("2 tests passed ✅  (pytest would find these automatically)")

---

## Now Take the Real Quiz

```bash
python weekly_quiz.py        # scored /15 — aim for 11+
python weekly_homework.py    # Typed Sensor Data Pipeline project
pytest weekly_homework.py -v # the project's own test suite
```

> **Next:** Week 09 — Project 1: **Task Manager CLI (`taskman`)**.
> Fundamentals are done — from here on, you build. 🚀